In [ ]:
from math import ceil


def calculate_vllm_vram(model, sequence_length, batch_size, precision_bytes):
    # 1. Base Model Memory
    base_memory = model.parameters * precision_bytes * model.overhead_factor

    # 2. KV-Cache with vLLM optimizations
    # Block allocation
    block_size = model.vllm_optimizations.block_size  # typically 16
    blocks_needed = ceil(sequence_length / block_size)
    effective_sequence_length = blocks_needed * block_size

    # GQA support
    kv_heads = model.architecture.kv_heads or model.architecture.attention_heads
    head_dim = model.architecture.head_dim or (model.architecture.hidden_size / model.architecture.attention_heads)

    # Core KV calculation
    kv_cache_base = (2 *                           # keys + values
                     model.architecture.layers *   # transformer layers
                     kv_heads *                     # KV heads (not attention heads for GQA!)
                     head_dim *                     # dimension per head
                     effective_sequence_length *   # tokens (block-aligned)
                     batch_size *                   # concurrent requests
                     precision_bytes)               # bytes per parameter

    # Memory pool overhead
    memory_pool_overhead = kv_cache_base * model.vllm_optimizations.memory_pool_overhead
    kv_cache_total = kv_cache_base + memory_pool_overhead

    # 3. Activation Memory
    activation_multiplier = model.vram_requirements.activation_multiplier  # typically 1.5
    activation_memory = (model.architecture.hidden_size *
                        sequence_length *
                        batch_size *
                        precision_bytes *
                        activation_multiplier)

    # 4. System Overhead
    system_overhead = (base_memory + kv_cache_total) * 0.1

    # Total VRAM
    total_vram = base_memory + kv_cache_total + activation_memory + system_overhead

    return total_vram


# Flexible ModelStub for different model configurations
class ModelStub:
    def __init__(self,
                 parameters=7_240_000_000,      # Total model parameters
                 overhead_factor=1.12,          # Model loading overhead
                 layers=32,                     # Number of transformer layers
                 hidden_size=4096,              # Hidden dimension size
                 attention_heads=32,            # Number of attention heads
                 kv_heads=8,                    # Number of KV heads (None for standard MHA)
                 head_dim=None,                 # Dimension per head (None to calculate)
                 block_size=16,                 # vLLM block size
                 memory_pool_overhead=0.15,     # vLLM memory pool overhead
                 activation_multiplier=1.4):    # Activation scaling factor

        self.parameters = parameters
        self.overhead_factor = overhead_factor

        # Architecture configuration
        self.architecture = type('Architecture', (), {
            'layers': layers,
            'hidden_size': hidden_size,
            'attention_heads': attention_heads,
            'kv_heads': kv_heads,              # Can be None for standard MHA
            'head_dim': head_dim               # Can be None to auto-calculate
        })()

        # vLLM optimization settings
        self.vllm_optimizations = type('VLLMOptimizations', (), {
            'block_size': block_size,
            'memory_pool_overhead': memory_pool_overhead
        })()

        # VRAM requirement settings
        self.vram_requirements = type('VRAMRequirements', (), {
            'activation_multiplier': activation_multiplier
        })()

In [ ]:
# Example 1: Default Mistral 7B with GQA
model_mistral = ModelStub()
print("Mistral 7B (GQA enabled):")
total_vram_bytes = calculate_vllm_vram(model_mistral, 500, 1, 2)
print(f"VRAM required: {total_vram_bytes / (1024**3):.2f} GB\n")

# Example 2: Llama 2 7B without GQA (standard MHA)
model_llama2 = ModelStub(
    parameters=7_000_000_000,    # 7B parameters
    overhead_factor=1.15,        # Slightly different overhead
    kv_heads=None,              # No GQA - will use attention_heads
    head_dim=None,              # Will be calculated as hidden_size/attention_heads
    activation_multiplier=1.5    # Different activation scaling
)
print("Llama 2 7B (standard MHA):")
total_vram_bytes = calculate_vllm_vram(model_llama2, 500, 1, 2)
print(f"VRAM required: {total_vram_bytes / (1024**3):.2f} GB\n")

# Example 3: Custom 13B model with specific GQA configuration
model_custom = ModelStub(
    parameters=13_000_000_000,   # 13B parameters
    layers=40,                   # More layers
    hidden_size=5120,            # Larger hidden size
    attention_heads=40,          # More attention heads
    kv_heads=8,                 # GQA: fewer KV heads
    head_dim=128,               # Explicit head dimension
    activation_multiplier=1.6    # Higher activation scaling
)
print("Custom 13B model with GQA:")
total_vram_bytes = calculate_vllm_vram(model_custom, 1000, 4, 2)
print(f"VRAM required: {total_vram_bytes / (1024**3):.2f} GB")